<a href="https://colab.research.google.com/github/AndrewNK1/BigData-Market-Basket-Analysis/blob/main/Big_data_Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Made by: Andrew Nabil Karam

In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("InstacartAnalysis").getOrCreate()

In [ ]:
!pip install kagglehub
import kagglehub

# Download latest version of the dataset
path = kagglehub.dataset_download("psparks/instacart-market-basket-analysis")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'instacart-market-basket-analysis' dataset.
Path to dataset files: /kaggle/input/instacart-market-basket-analysis


In [ ]:
import os

# Define the paths to the specific CSV files
orders_path = os.path.join(path, "orders.csv")
products_path = os.path.join(path, "products.csv")
order_products_path = os.path.join(path, "order_products__prior.csv")
departments_path = os.path.join(path, "departments.csv")
aisles_path = os.path.join(path, "aisles.csv")

# Load into Spark DataFrames with quote/escape handling to prevent malformed casts
# Using quote='"' and escape='"' ensures strings with commas or quotes are handled correctly.
df_orders = spark.read.csv(orders_path, header=True, inferSchema=True, quote='"', escape='"')
df_products = spark.read.csv(products_path, header=True, inferSchema=True, quote='"', escape='"')
df_order_products = spark.read.csv(order_products_path, header=True, inferSchema=True, quote='"', escape='"')
df_departments = spark.read.csv(departments_path, header=True, inferSchema=True, quote='"', escape='"')
df_aisles = spark.read.csv(aisles_path, header=True, inferSchema=True, quote='"', escape='"')

# Validation: Check the first few rows
df_orders.show(5)

+--------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------+--------+------------+---------+-----------------+----------------------+
| 2539329|      1|   prior|           1|        2|                8|                  NULL|
| 2398795|      1|   prior|           2|        3|                7|                  15.0|
|  473747|      1|   prior|           3|        3|               12|                  21.0|
| 2254736|      1|   prior|           4|        4|                7|                  29.0|
|  431534|      1|   prior|           5|        4|               15|                  28.0|
+--------+-------+--------+------------+---------+-----------------+----------------------+
only showing top 5 rows


### Step 1: Data Exploration and Cleaning
We will check for null values and duplicates in each of our DataFrames.

In [ ]:
from pyspark.sql.functions import col, count, when

dataframes = {
    "orders": df_orders,
    "products": df_products,
    "order_products": df_order_products,
    "departments": df_departments,
    "aisles": df_aisles
}

# Dictionary to store cleaned dataframes
cleaned_dataframes = {}

for name, df in dataframes.items():
    print(f"--- Analysis for: {name} ---")

    # 1. Count Nulls
    null_counts = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
    print("Null counts per column:")
    null_counts.show()

    # 2. Check for duplicates
    total_count = df.count()
    distinct_count = df.distinct().count()
    print(f"Total rows: {total_count}, Distinct rows: {distinct_count}")

    # 3. Fill Nulls with 0 instead of dropping
    # This preserves rows like the first order for a user in the orders table
    df_filled = df.na.fill(0)
    cleaned_dataframes[name] = df_filled

    print(f"Filled null values with 0 in {name}.")
    print(f"Row count for {name}: {df_filled.count()}")
    print("\n")

# Update global variables with cleaned data
df_orders = cleaned_dataframes['orders']
df_products = cleaned_dataframes['products']
df_order_products = cleaned_dataframes['order_products']
df_departments = cleaned_dataframes['departments']
df_aisles = cleaned_dataframes['aisles']

--- Analysis for: orders ---
Null counts per column:
+--------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------+--------+------------+---------+-----------------+----------------------+
|       0|      0|       0|           0|        0|                0|                206209|
+--------+-------+--------+------------+---------+-----------------+----------------------+

Total rows: 3421083, Distinct rows: 3421083
Filled null values with 0 in orders.
Row count for orders: 3421083


--- Analysis for: products ---
Null counts per column:
+----------+------------+--------+-------------+
|product_id|product_name|aisle_id|department_id|
+----------+------------+--------+-------------+
|         0|           0|       0|            0|
+----------+------------+--------+-------------+

Total rows: 49688, Distinct rows: 49688
Filled null values with 0 in pro

### Step 2: Creating a Unified Dataset
Now we join all tables together to create a single view of the Instacart transactions.

In [ ]:
# 1. Join Products with Aisles and Departments
df_merged_products = df_products.join(df_aisles, on="aisle_id", how="left") \
                                .join(df_departments, on="department_id", how="left")

# 2. Join Order_Products with the enriched Products info
df_full_prior = df_order_products.join(df_merged_products, on="product_id", how="left")

# 3. Join with Orders to get user and time information
df_unified = df_full_prior.join(df_orders, on="order_id", how="left")

# Validation: Check schema and a few records
print("Unified Dataset Schema:")
df_unified.printSchema()

print("Unified Dataset Preview:")
df_unified.limit(5).show()

print(f"Total records in unified dataset: {df_unified.count()}")

Unified Dataset Schema:
root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)
 |-- department_id: integer (nullable = true)
 |-- aisle_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle: string (nullable = true)
 |-- department: string (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- eval_set: string (nullable = true)
 |-- order_number: integer (nullable = true)
 |-- order_dow: integer (nullable = true)
 |-- order_hour_of_day: integer (nullable = true)
 |-- days_since_prior_order: double (nullable = true)

Unified Dataset Preview:
+--------+----------+-----------------+---------+-------------+--------+--------------------+--------------------+----------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|product_id|add_to_cart_order|reordered|department_id|aisle_id|        p

In [ ]:
from pyspark.sql.functions import avg, count, col, mean
from pyspark.sql.window import Window

# 1. Total items per order
df_unified = df_unified.withColumn("total_items_per_order", count("product_id").over(Window.partitionBy("order_id")))

# 2. Reorder ratio per order
# Calculate reorder count per order first
df_unified = df_unified.withColumn("reorder_count_per_order", count(when(col("reordered") == 1, 1)).over(Window.partitionBy("order_id")))
df_unified = df_unified.withColumn("reorder_ratio", col("reorder_count_per_order") / col("total_items_per_order"))

# 3. Customer order frequency (Average days since prior order per user)
df_unified = df_unified.withColumn("avg_order_frequency", avg("days_since_prior_order").over(Window.partitionBy("user_id")))

# Validation: Display columns for a sample user
df_unified.select("user_id", "order_id", "total_items_per_order", "reorder_ratio", "avg_order_frequency").distinct().show(10)

+-------+--------+---------------------+------------------+-------------------+
|user_id|order_id|total_items_per_order|     reorder_ratio|avg_order_frequency|
+-------+--------+---------------------+------------------+-------------------+
|      1|  473747|                    5|               0.6|  18.54237288135593|
|      1| 2398795|                    6|               0.5|  18.54237288135593|
|      1| 3367565|                    4|               1.0|  18.54237288135593|
|      1| 2550362|                    9|0.6666666666666666|  18.54237288135593|
|      1| 2254736|                    5|               1.0|  18.54237288135593|
|      1|  550135|                    5|               1.0|  18.54237288135593|
|      1| 3108588|                    6|0.6666666666666666|  18.54237288135593|
|      1| 2539329|                    5|               0.0|  18.54237288135593|
|      1|  431534|                    8|             0.625|  18.54237288135593|
|      1| 2295261|                    6|

### Step 3: Business Queries

First, we register our unified DataFrame as a temporary view to enable SparkSQL queries.

In [ ]:
df_unified.createOrReplaceTempView("instacart_data")

#### a) Top 10 Departments by Order Volume

**Business Interpretation:** This identifies the 'powerhouse' departments that drive the most traffic. Decision-makers can use this to prioritize inventory space, marketing spend, and supply chain logistics for these high-volume categories.

In [ ]:
# SparkSQL Version
spark.sql("""
    SELECT department, COUNT(product_id) as total_orders
    FROM instacart_data
    GROUP BY department
    ORDER BY total_orders DESC
    LIMIT 10
""").show()

# DataFrame Version
df_unified.groupBy("department") \
    .count() \
    .withColumnRenamed("count", "total_orders") \
    .orderBy(col("total_orders").desc()) \
    .limit(10) \
    .show()

+---------------+------------+
|     department|total_orders|
+---------------+------------+
|        produce|     9479291|
|     dairy eggs|     5414016|
|         snacks|     2887550|
|      beverages|     2690129|
|         frozen|     2236432|
|         pantry|     1875577|
|         bakery|     1176787|
|   canned goods|     1068058|
|           deli|     1051249|
|dry goods pasta|      866627|
+---------------+------------+

+---------------+------------+
|     department|total_orders|
+---------------+------------+
|        produce|     9479291|
|     dairy eggs|     5414016|
|         snacks|     2887550|
|      beverages|     2690129|
|         frozen|     2236432|
|         pantry|     1875577|
|         bakery|     1176787|
|   canned goods|     1068058|
|           deli|     1051249|
|dry goods pasta|      866627|
+---------------+------------+



#### b) Reorder Rate per Department & Customer Loyalty

**Business Interpretation:** A high reorder rate indicates strong customer loyalty or habit-forming behavior. Departments with high loyalty are reliable revenue anchors, while low loyalty departments might need quality improvements or personalized promotions to encourage repeat purchases.

In [ ]:
# SparkSQL Version
spark.sql("""
    SELECT department, AVG(reordered) as reorder_rate
    FROM instacart_data
    GROUP BY department
    ORDER BY reorder_rate DESC
""").show()

# DataFrame Version
df_unified.groupBy("department") \
    .agg(avg("reordered").alias("reorder_rate")) \
    .orderBy(col("reorder_rate").desc()) \
    .show()

+---------------+-------------------+
|     department|       reorder_rate|
+---------------+-------------------+
|     dairy eggs| 0.6699686517365298|
|      beverages| 0.6534601128793452|
|        produce| 0.6499125303780631|
|         bakery| 0.6281408615152955|
|           deli| 0.6077190085317561|
|           pets| 0.6012852523433343|
|         babies| 0.5789708401564881|
|           bulk|  0.577039886616724|
|         snacks| 0.5741798410417136|
|        alcohol| 0.5699237455756818|
|   meat seafood| 0.5676744281178281|
|      breakfast| 0.5609221936133061|
|         frozen| 0.5418854675661947|
|dry goods pasta| 0.4610761030985649|
|   canned goods| 0.4574049349379903|
|          other| 0.4079799399300102|
|      household|0.40217770954666926|
|        missing| 0.3958493021910478|
|  international|0.36922894081031593|
|         pantry| 0.3467205025440171|
+---------------+-------------------+
only showing top 20 rows
+---------------+-------------------+
|     department|       r

#### c) Peak Shopping Hours

**Business Interpretation:** Identifying peak hours allows the business to optimize staffing for picking and packing, schedule server maintenance during low-traffic periods, and adjust delivery windows to manage expectations during high-demand surges.

In [ ]:
from pyspark.sql.functions import countDistinct

# SparkSQL Version
spark.sql("""
    SELECT order_hour_of_day, COUNT(DISTINCT order_id) as order_count
    FROM instacart_data
    GROUP BY order_hour_of_day
    ORDER BY order_count DESC
""").show()

# DataFrame Version
df_unified.groupBy("order_hour_of_day") \
    .agg(countDistinct("order_id").alias("order_count")) \
    .orderBy(col("order_count").desc()) \
    .show()

+-----------------+-----------+
|order_hour_of_day|order_count|
+-----------------+-----------+
|               10|     271885|
|               11|     268006|
|               15|     266132|
|               14|     265556|
|               13|     261174|
|               12|     256206|
|               16|     255949|
|                9|     243496|
|               17|     214080|
|               18|     170998|
|                8|     168321|
|               19|     131620|
|               20|      98109|
|                7|      86656|
|               21|      73436|
|               22|      57540|
|               23|      37613|
|                6|      28792|
|                0|      21372|
|                1|      11596|
+-----------------+-----------+
only showing top 20 rows
+-----------------+-----------+
|order_hour_of_day|order_count|
+-----------------+-----------+
|               10|     271885|
|               11|     268006|
|               15|     266132|
|              

#### d) Average Basket Size

**Business Interpretation:** Basket size reveals the average volume of products per transaction. Increasing this number through cross-selling or "buy more, save more" promotions can directly increase revenue without needing to acquire new customers.

In [ ]:
# SparkSQL Version
spark.sql("""
    SELECT AVG(total_items_per_order) as avg_basket_size
    FROM (SELECT DISTINCT order_id, total_items_per_order FROM instacart_data)
""").show()

# DataFrame Version
from pyspark.sql.functions import mean
df_unified.select("order_id", "total_items_per_order").distinct() \
    .select(mean("total_items_per_order").alias("avg_basket_size")) \
    .show()

+------------------+
|   avg_basket_size|
+------------------+
|10.088883421247614|
+------------------+

+------------------+
|   avg_basket_size|
+------------------+
|10.088883421247614|
+------------------+



#### e) Top 10 Most Frequently Reordered Products

**Business Interpretation:** Products with high reorder rates are likely staples (milk, bananas, etc.). These are the items customers rely on; ensuring they are never out of stock is critical for retention.

In [ ]:
# SparkSQL Version
spark.sql("""
    SELECT product_name, SUM(reordered) as total_reorders
    FROM instacart_data
    GROUP BY product_name
    ORDER BY total_reorders DESC
    LIMIT 10
""").show()

# DataFrame Version
from pyspark.sql.functions import sum as _sum
df_unified.groupBy("product_name") \
    .agg(_sum("reordered").alias("total_reorders")) \
    .orderBy(col("total_reorders").desc()) \
    .limit(10) \
    .show()

+--------------------+--------------+
|        product_name|total_reorders|
+--------------------+--------------+
|              Banana|        398609|
|Bag of Organic Ba...|        315913|
|Organic Strawberries|        205845|
|Organic Baby Spinach|        186884|
|Organic Hass Avocado|        170131|
|     Organic Avocado|        134044|
|  Organic Whole Milk|        114510|
|         Large Lemon|        106255|
| Organic Raspberries|        105409|
|        Strawberries|         99802|
+--------------------+--------------+

+--------------------+--------------+
|        product_name|total_reorders|
+--------------------+--------------+
|              Banana|        398609|
|Bag of Organic Ba...|        315913|
|Organic Strawberries|        205845|
|Organic Baby Spinach|        186884|
|Organic Hass Avocado|        170131|
|     Organic Avocado|        134044|
|  Organic Whole Milk|        114510|
|         Large Lemon|        106255|
| Organic Raspberries|        105409|
|        St

#### f) Percentage Contribution of Each Department to Total Sales

**Business Interpretation:** This helps identify the dominant categories that sustain the business. If a single department (like Produce) accounts for a massive percentage of sales, the company's risk is tied heavily to that supply chain.
The dominant categories are produce, dairy eggs ,snacks and beverages.


In [ ]:
total_count = df_unified.count()

# SparkSQL Version
spark.sql(f"""
    SELECT department,
           (COUNT(product_id) * 100.0 / {total_count}) as percentage_contribution
    FROM instacart_data
    GROUP BY department
    ORDER BY percentage_contribution DESC
""").show()

# DataFrame Version
df_unified.groupBy("department") \
    .count() \
    .withColumn("percentage_contribution", (col("count") * 100 / total_count)) \
    .orderBy(col("percentage_contribution").desc()) \
    .show()

+---------------+-----------------------+
|     department|percentage_contribution|
+---------------+-----------------------+
|        produce|          29.2259606741|
|     dairy eggs|          16.6921575364|
|         snacks|           8.9027146381|
|      beverages|           8.2940384848|
|         frozen|           6.8952281012|
|         pantry|           5.7826623999|
|         bakery|           3.6281965164|
|   canned goods|           3.2929700234|
|           deli|           3.2411455596|
|dry goods pasta|           2.6719304873|
|      household|           2.2774090876|
|      breakfast|           2.1876990262|
|   meat seafood|           2.1857319842|
|  personal care|           1.3785418355|
|         babies|           1.3066399782|
|  international|           0.8301441099|
|        alcohol|           0.4738659518|
|           pets|           0.3012965612|
|        missing|           0.2131835652|
|          other|           0.1118901550|
+---------------+-----------------

**4.User Segmentation**:
We segment users by their total number of orders:
- **Low Activity**: < 10 orders
- **Medium Activity**: 10 - 40 orders
- **High Activity**: > 40 orders

In [ ]:
from pyspark.sql.functions import col, when, max as _max

# 1. Get max order number per user
df_user_activity = df_orders.groupBy("user_id").agg(_max("order_number").alias("total_user_orders"))

# 2. Segment using when/otherwise
df_segments = df_user_activity.withColumn("segment",
    when(col("total_user_orders") < 10, "Low Activity")
    .when((col("total_user_orders") >= 10) & (col("total_user_orders") <= 40), "Medium Activity")
    .otherwise("High Activity")
)

# 3. Count users per segment
df_segments.groupBy("segment").count().show()

+---------------+-----+
|        segment|count|
+---------------+-----+
|   Low Activity|95481|
|Medium Activity|92826|
|  High Activity|17902|
+---------------+-----+

